# FloodDepth

[Codility task](https://app.codility.com/programmers/trainings/1/flood_depth/) - Exercise 1, 2015 Contest

`A[i]` is the altitude of block `i`. After a flood every low-lying area holds as much water as possible; the outside of the array has altitude 0 and drains everything. `solution(A)` returns the maximum water depth.

| A | result |
|---|--------|
| `[1, 3, 2, 1, 2, 1, 5, 3, 3, 4, 2]` | 2 (blocks 3 and 5 hold 2) |
| `[5, 8]` | 0 |

Assumptions: `N` in `[1..100,000]`, values in `[1..100,000,000]`.

## Idea

Water above block `i` cannot rise above the highest block on its left nor above the highest block on its right:

`level[i] = min(max(A[0..i]), max(A[i..N-1]))`, depth `= level[i] - A[i]`.

Because each maximum includes `A[i]` itself, the depth is never negative. The first and last blocks always have depth 0 (one of their maxima is themselves), which models the draining outside.

Two passes build the prefix maxima, one more computes the answer: O(N) time and space.

In [ ]:
def solution(A):
    n = len(A)
    right = [0] * n                  # right[i] = max(A[i..n-1])
    highest = 0
    for i in range(n - 1, -1, -1):
        highest = max(highest, A[i])
        right[i] = highest

    best = 0
    left = 0                         # running max(A[0..i])
    for i in range(n):
        left = max(left, A[i])
        best = max(best, min(left, right[i]) - A[i])
    return best

## Official examples

In [ ]:
assert solution([1, 3, 2, 1, 2, 1, 5, 3, 3, 4, 2]) == 2
assert solution([5, 8]) == 0
print("examples OK")

## Edge cases

In [ ]:
assert solution([7]) == 0                                  # one block
assert solution([1, 2, 3, 4, 5]) == 0                      # increasing: everything drains to the left
assert solution([5, 4, 3, 2, 1]) == 0                      # decreasing
assert solution([5, 1, 5]) == 4                            # one deep lake
assert solution([5, 1, 1, 1, 5]) == 4                      # a wide lake: depth is the same
assert solution([3, 1, 2, 1, 3]) == 2                      # level 3 everywhere inside
assert solution([5, 1, 4, 1, 3]) == 3                      # the right wall (3) limits the second lake; the first lake level is 4
assert solution([2, 2, 2, 2]) == 0                         # flat
assert solution([10**8, 1, 10**8]) == 10**8 - 1            # largest values
assert solution([1, 100, 1, 100, 1]) == 99
print("edge cases OK")

## Independent reference: relaxation of the water level

The reference does not use prefix maxima. It starts with an absurdly high water level everywhere and repeatedly applies the physical rule *"the water in a block can only stay as high as the lower of its neighbours, but never below the rock"*:

`level[i] = max(A[i], min(level[i-1], level[i+1]))`, with level 0 outside.

Iterating until nothing changes converges to the greatest stable state, the flooded landscape.

In [ ]:
import random


def brute(A):
    n = len(A)
    level = [10**9] * n
    changed = True
    while changed:
        changed = False
        for i in range(n):
            left = level[i - 1] if i > 0 else 0
            right = level[i + 1] if i < n - 1 else 0
            new = max(A[i], min(left, right))
            if new != level[i]:
                level[i] = new
                changed = True
    return max(level[i] - A[i] for i in range(n))


rng = random.Random(2)
for _ in range(4000):
    n = rng.randint(1, 14)
    A = [rng.randint(1, 8) for _ in range(n)]
    assert solution(A) == brute(A), A
print("matches the relaxation reference on 4000 random landscapes")

## Performance

In [ ]:
import time

N = 100_000
cases = {
    "random": [rng.randint(1, 10**8) for _ in range(N)],
    "valley": list(range(N // 2, 0, -1)) + list(range(1, N // 2 + 1)),
    "mountain": list(range(1, N // 2 + 1)) + list(range(N // 2, 0, -1)),
    "flat": [5] * N,
}
for name, A in cases.items():
    start = time.perf_counter()
    result = solution(A)
    print(f"{name:10s} -> {result:>10} in {time.perf_counter() - start:.4f}s")

## Notes

- This is the "trapping rain water" problem; the classic O(N) answer is *min of the two running maxima minus the height*.
- The outside having altitude 0 is what makes the borders hold nothing.
- A two-pointer version needs O(1) extra space; the two-array version is easier to read and to test.